<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Fine-tune a small transformer

**[Fine-tune a small transformer](https://learning.nextia-ai.com/courses/deep-learning/m08/fine-tune-a-small-transformer/)** · Deep Learning and Transformers Explained · Module 8, case study 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** route real support messages to 77 intents with four models under one split and one metric: word counts (TF-IDF and logistic regression), the course's tiny transformer trained from scratch, a pretrained DistilBERT used as a frozen feature maker, and DistilBERT fine-tuned. You measure what pretraining buys when labelled data is scarce.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 2 minutes on the course's computer; allow up to 10 minutes on a free Colab CPU. |
| **Needs** | An internet connection for the setup cells (about 270 MB for DistilBERT, 1 MB for the data, 2 MB for the course's code and tickets). No account, no graphics card. About 2 GB of free memory. |
| **Smaller than the page** | To fit in 10 minutes, the notebook fine-tunes on 500 messages (and 100 in a task), not on all 9,003. The page's numbers for all 9,003 messages took about 8 minutes of training on a CPU. |
| **You should know** | The training loop and early stopping from [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/) and [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/), the transformer block from [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/), masked-token prediction from [Training objectives](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/), and loading a pretrained model from [Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/). |
| **Data** | BANKING77 (PolyAI, 2020), CC BY 4.0: 13,083 short messages to a bank's online support, 77 intents. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled; Hugging Face transformers 5.19.0 |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/fine-tune-a-small-transformer/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L03-fine-tune-a-small-transformer/fine-tune-a-small-transformer-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: Hugging Face transformers

Run the next cell. It loads Hugging Face `transformers`, the library that downloads published models and builds them as PyTorch networks, and installs it if it is missing. Colab and Kaggle usually have it already. You should see `transformers` and its version. The course used 5.19.0; version 4 should also work, but it is not tested.

In [ ]:
try:
    import transformers
except ImportError:  # not installed: install it
    %pip install -q --disable-pip-version-check "transformers>=5,<6" safetensors
    import transformers
transformers.logging.set_verbosity_error()   # hide the long load report of a new classification head
print("transformers", transformers.__version__)

## Setup: get BANKING77

Run the next cell. It downloads the two files of **BANKING77** (about 1.1 MB) from the publisher's GitHub repository, at one fixed version, into `ticket-router/data/banking77/`, and checks each file's checksum.

You should see `Ready: train.csv 10,003 messages, test.csv 3,080 messages.`

Data: "BANKING77" by PolyAI (Casanueva et al., 2020), https://github.com/PolyAI-LDN/task-specific-datasets, licensed under CC BY 4.0. We changed nothing in the files.

In [ ]:
import csv
BANKING_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
BANKING_SHA256 = {
    "train.csv": "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b",
    "test.csv": "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"
}
BANKING = Path("data/banking77")
BANKING.mkdir(parents=True, exist_ok=True)
for name, expected in BANKING_SHA256.items():
    path = BANKING / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(BANKING_URL + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder data/banking77 and run again.")

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

raw_train, raw_test = read_csv(BANKING / "train.csv"), read_csv(BANKING / "test.csv")
print(f"Ready: train.csv {len(raw_train):,} messages, test.csv {len(raw_test):,} messages.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

Nadia leads the support team of an online bank. Each chat message must reach the right handler: one of 77 **intents**, such as `card_arrival`, `top_up_failed` or `cancel_transfer`. She asks Kofi, the bank's machine learning engineer, whether a pretrained model is worth it, most of all for a new product, when only a few hundred messages have a label.

Look at the size, the intents and a few messages.

> **Predict.** Will every intent have about the same number of messages?

In [ ]:
from collections import Counter
print(len(raw_train), "train |", len(raw_test), "test | columns:", list(raw_train[0].keys()))
counts = Counter(r["category"] for r in raw_train)
print("intents:", len(counts), "| messages per intent in train: min", min(counts.values()), "max", max(counts.values()))
print("test, per intent:", sorted(set(Counter(r["category"] for r in raw_test).values())))
for intent in ("top_up_failed", "top_up_reverted", "pending_top_up"):
    print(f"{intent:16}", [r["text"] for r in raw_train if r["category"] == intent][:2])

> **Check.** You should see `10003 train | 3080 test`, `intents: 77`, a minimum of 35 and a maximum of 187 messages per intent in train, and exactly 40 per intent in test.

Three intents about a top-up share almost every word: "top-up", "money", "not". Only the rest of the message says whether the top-up failed, came back, or is still waiting. That is where word counts may struggle.

## 2. One split, one metric

The official test file stays sealed until the end. Kofi holds out 1,000 messages of the train file as a **validation set**, the same share of each intent, to compare the models. The metric is **accuracy** on these 1,000 messages. With 77 intents, always answering the most common intent is right only 1.9% of the time.

For the learning curve, `subset()` takes a sample of the training messages with the same share of each intent. Sizes 100 and 500 imitate a new product with few labels.

In [ ]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(raw_train, test_size=1000, stratify=[r["category"] for r in raw_train], random_state=0)
test = raw_test
INTENTS = sorted(counts)
y_of = lambda rows: np.array([INTENTS.index(r["category"]) for r in rows])
texts = lambda rows: [r["text"] for r in rows]
y_valid = y_of(valid)

def subset(rows, n, seed=0):
    """n rows with the same share of each intent; all rows when n >= len(rows)."""
    if n >= len(rows):
        return list(rows)
    return train_test_split(rows, train_size=n, stratify=[r["category"] for r in rows], random_state=seed)[0]

print(len(train), "train |", len(valid), "valid |", len(test), "test (sealed)")
print("intents in a sample of 100:", len({r["category"] for r in subset(train, 100)}))

> **Check.** You should see `9003 train | 1000 valid | 3080 test (sealed)` and `intents in a sample of 100: 75`: most intents get one message, a few get two, and 2 intents get none.

## 3. Bag of words: TF-IDF and logistic regression

The baseline is the machine learning course's router: TF-IDF word weights and logistic regression, with the course's word tokens (`tokenize` from `ticketnet.py`). A small grid on the validation set chose `min_df=1` and `C=100` (the page shows the grid).

> **Predict.** With 100 labelled messages, about 1 per intent, what share of the validation messages will it route right? And with all 9,003?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def fit_tfidf(texts_, y, min_df=1, C=100):
    return make_pipeline(TfidfVectorizer(token_pattern=None, tokenizer=tokenize, lowercase=False, min_df=min_df),
                         LogisticRegression(max_iter=3000, C=C)).fit(texts_, y)

results = {}
for n in (100, 500, len(train)):
    part = subset(train, n)
    model = fit_tfidf(texts(part), y_of(part))
    results[("TF-IDF", n)] = (model.predict(texts(valid)) == y_valid).mean()
    print(f"TF-IDF, {n:5} messages: valid accuracy {results[('TF-IDF', n)]:.3f}")

> **Check.** You should see about `0.305`, `0.608` and `0.880`.

> **Your turn.** Fit the same model on a sample of 2,000 messages and put its validation accuracy in `tfidf_2000`. Then run the check cell.

In [ ]:
part = subset(train, 2000)
tfidf_2000 = (fit_tfidf(texts(part), y_of(part)).predict(texts(valid)) == y_valid).mean()
results[("TF-IDF", 2000)] = tfidf_2000
print(round(tfidf_2000, 3))

In [ ]:
expect('tfidf_2000', tfidf_2000 if tfidf_2000 is ... else round(float(tfidf_2000), 3), 0.793, tolerance=0.002)

## 4. From scratch: the course's tiny transformer

The course's `TinyTransformer` (embeddings, positions, 2 blocks) starts from random weights. It must learn English and the intents from the labelled messages alone. `ticketnet.py` builds it with 5 outputs for Larkfield's teams; here the last layer gets 77. Training keeps the epoch with the lowest validation loss (**early stopping**).

> **Predict.** On 500 messages, will it beat TF-IDF (60.8%)?

In [ ]:
import copy, math
from torch import nn

def tiny_transformer(vocab_size, n_classes, seed=0):
    torch.manual_seed(seed)
    model = TinyTransformer(vocab_size)
    model.out = nn.Linear(64, n_classes)          # 77 intents instead of 5 teams
    return model

def fit_early_stopping(model, x_tr, y_tr, x_va, y_va, epochs, batch_size=32, lr=1e-3, seed=0):
    """The five steps, and keep the weights of the epoch with the lowest validation loss."""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best = (math.inf, 0, None, 0.0)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(x_tr))
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(x_tr[idx]), y_tr[idx])
            loss.backward()
            opt.step()
        vl, va = evaluate(model, x_va, y_va)       # evaluation mode, no gradients
        if vl < best[0]:
            best = (vl, epoch, copy.deepcopy(model.state_dict()), va)
    model.load_state_dict(best[2])
    return best[1], best[3]

part = subset(train, 500)
vocab = Vocab(texts(part))
x_tr, x_va = token_ids(part, vocab), token_ids(valid, vocab)
show_shape("x_tr", x_tr)
tiny = tiny_transformer(len(vocab), len(INTENTS))
print("parameters:", count_parameters(tiny))
best_epoch, results[("tiny transformer", 500)] = fit_early_stopping(
    tiny, x_tr, torch.tensor(y_of(part)), x_va, torch.tensor(y_valid), epochs=60)
print(f"tiny transformer, 500 messages: best epoch {best_epoch}, valid accuracy {results[('tiny transformer', 500)]:.3f}")

> **Check.** You should see a vocabulary of 458 entries (the second number of `x_tr`'s shape is the longest message), `parameters: 142093`, and a validation accuracy of about `0.327`, far below TF-IDF. It takes about a minute.

500 short messages hold about 6,000 words. That is far too little to learn what words mean. On the page, the same network trained on all 9,003 messages reaches 79.6%.

## 5. The pretrained encoder: DistilBERT

**DistilBERT** was pretrained by its makers with masked-token prediction on books and English Wikipedia. It reads the whole message in both directions: an **encoder**. The next cell downloads it (about 270 MB, once) and shows how its tokenizer cuts a message into subword pieces.

In [ ]:
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

MODEL = "distilbert/distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"  # the exact version of the files that the course used
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
encoder = AutoModel.from_pretrained(MODEL, revision=REVISION)   # the pretrained network, without a head
encoder.eval()
print("encoder parameters:", count_parameters(encoder))
print(tokenizer.convert_ids_to_tokens(tokenizer("My top-up was reverted?")["input_ids"]))

> **Check.** You should see `encoder parameters: 66362880` and the pieces `['[CLS]', 'my', 'top', '-', 'up', 'was', 'reverted', '?', '[SEP]']`. `[CLS]` and `[SEP]` mark the start and the end of a message.

## 6. Frozen: the encoder as a feature maker

The cheapest way to use a pretrained encoder: run each message through it once, with no training, and average the last layer's 768 numbers per token into one row per message. Then logistic regression learns the intents from these rows. Only the logistic regression is trained.

> **Predict.** Will 768 pretrained numbers per message beat TF-IDF's word weights with 100 messages?

In [ ]:
def encode(texts_, max_length=64):
    return tokenizer(texts_, padding=True, truncation=True, max_length=max_length, return_tensors="pt")

def features(texts_, batch_size=64):
    """The frozen encoder: the average of the last layer's token vectors (padding excluded), 768 per message."""
    out = []
    with torch.no_grad():
        for i in range(0, len(texts_), batch_size):
            b = encode(texts_[i:i + batch_size])
            h = encoder(**b).last_hidden_state                     # (batch, tokens, 768)
            m = b["attention_mask"].unsqueeze(-1).float()          # (batch, tokens, 1): 0 for padding
            out.append((h * m).sum(1) / m.sum(1))                  # (batch, 768)
    return torch.cat(out).numpy()

from sklearn.preprocessing import StandardScaler
part_2000 = subset(train, 2000)
X_2000, X_valid = features(texts(part_2000)), features(texts(valid))
print("features:", X_2000.shape, X_valid.shape)
row = {id(r): i for i, r in enumerate(part_2000)}
for n in (100, 500, 2000):
    part = subset(train, n)
    if n < 2000:  # the samples of 100 and 500 are not inside the sample of 2,000: compute their rows
        X = features(texts(part))
    else:
        X = X_2000
    frozen = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)).fit(X, y_of(part))
    results[("frozen DistilBERT", n)] = (frozen.predict(X_valid) == y_valid).mean()
    print(f"frozen DistilBERT, {n:5} messages: valid accuracy {results[('frozen DistilBERT', n)]:.3f}")

> **Check.** You should see `features: (2000, 768) (1000, 768)`, then about `0.265`, `0.606` and `0.775`. That is close to TF-IDF, not far above it: the frozen features were made for filling in masked words, not for telling `top_up_failed` from `top_up_reverted`.

## 7. Fine-tuned: train the pretrained weights a little more

**Fine-tuning** starts from the pretrained weights, adds a new output layer with 77 scores (the **head**), and trains everything with the course's five steps and a small learning rate (5e-05, from a small grid on the page). `AutoModelForSequenceClassification` builds DistilBERT with that new head; `labels=` makes it compute the cross-entropy loss itself.

This cell trains on 500 messages for 8 epochs. The page's run went on to 25 epochs and kept epoch 24 (72.3%); the notebook stops at 8 to save time. It takes about 2 to 4 minutes on a free Colab CPU.

> **Predict.** TF-IDF reached 60.8% with 500 messages. Where will the fine-tuned model land?

In [ ]:
def predict_proba(model, texts_, batch_size=64):
    model.eval()                                   # evaluation mode: dropout off
    out = []
    with torch.no_grad():
        for i in range(0, len(texts_), batch_size):
            out.append(torch.softmax(model(**encode(texts_[i:i + batch_size])).logits, -1))
    return torch.cat(out)

def fine_tune(model, texts_, y, epochs, batch_size=16, lr=5e-05, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    y = torch.as_tensor(y)
    for epoch in range(1, epochs + 1):
        model.train()                              # training mode: dropout on
        order = torch.randperm(len(texts_)).tolist()
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()                                                    # 1
            out = model(**encode([texts_[j] for j in idx]), labels=y[idx])     # 2 and 3
            out.loss.backward()                                                # 4
            opt.step()                                                         # 5
        print(f"epoch {epoch}: last batch loss {out.loss.item():.3f}")
    return model

part = subset(train, 500)
torch.manual_seed(0)
classifier = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION, num_labels=len(INTENTS))
print("parameters with the new head:", count_parameters(classifier))
fine_tune(classifier, texts(part), y_of(part), epochs=8)
p_valid = predict_proba(classifier, texts(valid))
results[("fine-tuned DistilBERT", 500)] = (p_valid.argmax(1).numpy() == y_valid).mean()
print(f"fine-tuned DistilBERT, 500 messages: valid accuracy {results[('fine-tuned DistilBERT', 500)]:.3f}")

> **Check.** You should see `parameters with the new head: 67012685`, a loss that falls over the epochs, and a validation accuracy of about `0.697` (a point or two more or less on another processor): the page's run had 69.7% after epoch 8.

## 8. The comparison

Put every result of this notebook into one table, one row per model, one column per size.

In [ ]:
import pandas as pd
table = pd.Series({k: round(float(v), 3) for k, v in results.items()}).unstack()
print(table.to_string())

> **Check.** The column 500 should read about: TF-IDF 0.608, tiny transformer 0.327, frozen DistilBERT 0.606, fine-tuned DistilBERT 0.697 (after 8 epochs).

On the page, with all 9,003 messages: TF-IDF 88.0%, tiny transformer 79.6%, frozen 87.0%, fine-tuned 91.6%.

## 9. New messages

These six messages are made up for the case study. The fourth and the fifth use exactly the same words in a different order.

In [ ]:
NEW = [
    "I topped up my account this morning but the money still isn't there.",
    "My new card still hasn't arrived and I ordered it two weeks ago.",
    "I was charged twice for the same coffee.",
    "The transfer went through, but the top-up failed.",
    "The top-up went through, but the transfer failed.",
    "Can I get my money back for something I bought online? The seller never sent it.",
]
p_new = predict_proba(classifier, NEW)
for text, p in zip(NEW, p_new):
    top = torch.topk(p, 2)
    print(text)
    print("   ", ", ".join(f"{INTENTS[int(i)]} {v:.3f}" for v, i in zip(top.values, top.indices)))

> **Check.** You should see two intents with a probability for each message. With only 500 training messages, some answers are unsure or wrong; the page shows the answers of the model trained on all messages. Compare the fourth and the fifth message: a bag of words gives these two messages the same input, so it must give them the same answer. The fine-tuned encoder reads the order, so its answers can differ.

## 10. Where the fine-tuned model goes wrong

Count the most common confusions on the validation set: true intent → predicted intent.

In [ ]:
pred = p_valid.argmax(1).numpy()
pairs = Counter((INTENTS[t], INTENTS[p]) for t, p in zip(y_valid, pred) if t != p)
for (t, p), n in pairs.most_common(6):
    print(f"{n:2}  {t} -> {p}")
confidence = p_valid.max(1).values.numpy()
print("median confidence, right:", round(float(np.median(confidence[pred == y_valid])), 2),
      "| wrong:", round(float(np.median(confidence[pred != y_valid])), 2))

> **Check.** The confusions are between intents that a person could also mix up, such as two kinds of top-up or card problems. The wrong answers have a much lower median confidence than the right ones. That makes a fallback possible: send unsure messages to a person.

> **Your turn.** Nadia wants a fallback: messages where the model's highest probability is below 0.5 go to a person. Compute two numbers from `confidence`, `pred` and `y_valid`: `share_to_person`, the share of validation messages below 0.5, and `accuracy_kept`, the accuracy on the other messages. Then run the check cell.

In [ ]:
keep = confidence >= 0.5
share_to_person = float((~keep).mean())
accuracy_kept = float((pred[keep] == y_valid[keep]).mean())
print(round(share_to_person, 3), round(accuracy_kept, 3))

In [ ]:
if share_to_person is Ellipsis or accuracy_kept is Ellipsis:
    print("Not yet: write both answers in the cell above, then run it again.")
elif abs(share_to_person - float((confidence < 0.5).mean())) > 1e-9:
    print("Not yet: share_to_person should be the share of messages whose highest probability is below 0.5.")
elif abs(accuracy_kept - float((pred[confidence >= 0.5] == y_valid[confidence >= 0.5]).mean())) > 1e-9:
    print("Not yet: accuracy_kept should use only the messages with a highest probability of 0.5 or more.")
else:
    print(f"Check passed: {share_to_person:.1%} of the messages go to a person; the router is right on {accuracy_kept:.1%} of the rest.")

With the model fine-tuned on all 9,003 messages, the page finds that 2.8% of the validation messages go to a person, and the router is right on 93.1% of the rest (against 91.6% on all of them) Your model, trained on 500 messages for 8 epochs, is much less sure, so the same threshold sends most messages to a person. A fallback trades the work of a person for fewer wrong answers; Nadia chooses the threshold from what each costs.

## Next

You trained and compared four routers on real messages under one split and one metric. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/fine-tune-a-small-transformer/) has the full learning curve, the costs, the test result, the same method on Larkfield's tickets, the responsible-use checks and the portfolio template. The next case study, [Train a tiny language model](https://learning.nextia-ai.com/courses/deep-learning/m08/train-a-tiny-language-model/), trains a small decoder from scratch to write text.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Train a tiny language model](https://learning.nextia-ai.com/courses/deep-learning/m08/train-a-tiny-language-model/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/fine-tune-a-small-transformer/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). BANKING77 by PolyAI (2020), CC BY 4.0. Larkfield's tickets are made up for this course by a language model.